In [1]:
!pip install scikit-learn

import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [4]:
fake = pd.read_csv('Fake.csv', on_bad_lines='skip', engine='python')
true = pd.read_csv('True.csv', on_bad_lines='skip', engine='python')

# Add labels
fake['label'] = 0
true['label'] = 1

# Combine
df = pd.concat([fake, true], axis=0)

# Select columns
df = df[['text', 'label']]

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (39700, 2)


,text,label
0,President Trump just exposed one of the bigges...,0
1,https://twitter.com/TEN_GOP/status/79389017105...,0
2,Isn t this just the height of hypocrisy? Hilla...,0
3,Donald Trump just got owned for comparing Hill...,0
4,WASHINGTON - Bernie Sanders won the U.S. presi...,1


In [5]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', ' ', text)
    return text

df['text'] = df['text'].apply(clean_text)

In [6]:
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [8]:
model = LinearSVC()
model.fit(X_train_vec, y_train)

LinearSVC()

In [9]:
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9919395465994962

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4198
           1       0.99      0.99      0.99      3742

    accuracy                           0.99      7940
   macro avg       0.99      0.99      0.99      7940
weighted avg       0.99      0.99      0.99      7940



In [10]:
def predict_news(text):
    cleaned = clean_text(text)
    vectorized = vectorizer.transform([cleaned])
    pred = model.predict(vectorized)

    return "Real News ✅" if pred[0] == 1 else "Fake News ❌"

# Test
print(predict_news("Breaking news: Government announces new scheme"))

Fake News ❌
